# Uplift Modelling with EconML and DoWhy

This notebook estimates the **heterogeneous treatment effect** of a *Buy One Get One* offer on customer conversion.

It compares two causal machine-learning estimators:

- **Causal Forest Double Machine Learning (`CausalForestDML`)**
- **Forest Doubly Robust Learner (`ForestDRLearner`)**

The workflow covers data preparation, causal estimation, uplift evaluation, cost-aware policy selection, off-policy revenue evaluation, and interpretable customer targeting.

> **Causal assumption:** the treatment must be randomized or conditionally unconfounded after controlling for the included features. For observational data, define a domain-informed causal graph and verify overlap before interpreting the estimates causally.


## 1. Install dependencies

The original notebook installed several unused packages. This version installs only the libraries used below.


In [ ]:
%%capture
%pip install -q "econml[dowhy]" scikit-uplift xgboost


## 2. Imports and configuration


In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from econml.cate_interpreter import SingleTreeCateInterpreter
from econml.dml import CausalForestDML
from econml.dr import ForestDRLearner
from sklearn.model_selection import train_test_split
from sklift.metrics import qini_auc_score, uplift_auc_score
from sklift.viz import plot_uplift_curve
from xgboost import XGBClassifier

%matplotlib inline

SEED = 24
TEST_SIZE = 0.20

np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 3. Load and validate the data

The source is the Hillstrom email-marketing dataset. This analysis keeps only:

- `No Offer` — control group (`0`)
- `Buy One Get One` — treatment group (`1`)

The `Discount` arm is excluded so that the treatment is binary.


In [ ]:
DATA_URL = (
    "https://drive.google.com/uc?id="
    "1vjqiiAWpBv6IO0Fd-Rz8BTgXrIZy1Q2w"
)

raw_df = pd.read_csv(DATA_URL)

required_columns = {
    "recency",
    "history",
    "used_discount",
    "used_bogo",
    "zip_code",
    "is_referral",
    "channel",
    "offer",
    "conversion",
}

missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = (
    raw_df.loc[raw_df["offer"].isin(["No Offer", "Buy One Get One"])]
    .copy()
    .reset_index(drop=True)
)

print(f"Rows retained: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head()


In [ ]:
# Data types and missing values
df.info()


In [ ]:
# Numerical summary
df.describe().T


In [ ]:
# Categorical summary
df.describe(include=["object"]).T


## 4. Preprocess the features

Categorical variables are one-hot encoded. One category per variable is dropped to avoid redundant columns.

The treatment coding is:

- `0`: No Offer
- `1`: Buy One Get One


In [ ]:
TREATMENT_MAP = {
    "No Offer": 0,
    "Buy One Get One": 1,
}

model_df = df.copy()
model_df["offer"] = model_df["offer"].map(TREATMENT_MAP).astype("int8")

model_df = pd.get_dummies(
    model_df,
    columns=["zip_code", "channel"],
    drop_first=True,
    dtype="int8",
)

if model_df.isna().any().any():
    missing = model_df.columns[model_df.isna().any()].tolist()
    raise ValueError(f"Missing values detected after preprocessing: {missing}")

model_df.head()


In [ ]:
Y = model_df["conversion"].astype(float)
T = model_df["offer"].astype(int)
X = model_df.drop(columns=["conversion", "offer"])

print(f"Feature matrix: {X.shape}")
print(f"Treatment rate: {T.mean():.2%}")
print(f"Overall conversion rate: {Y.mean():.2%}")


## 5. Create a reproducible holdout set

A random, treatment-stratified split is preferable to taking the first 20% of rows because the source data may have an ordering pattern. Stratification preserves the treatment proportion in both samples.


In [ ]:
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X,
    T,
    Y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=T,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Train treatment rate: {T_train.mean():.2%}")
print(f"Test treatment rate: {T_test.mean():.2%}")


## 6. First-stage nuisance models

EconML already performs cross-fitting. Running a full grid search inside every cross-fitting fold creates expensive nested cross-validation, so this notebook uses reproducible, regularized XGBoost classifiers.

`n_jobs=1` is intentional: EconML parallelizes the causal forests, and limiting each XGBoost model prevents CPU oversubscription.


In [ ]:
def make_binary_xgb(seed: int = SEED) -> XGBClassifier:
    # Regularized binary classifier for nuisance estimation.
    return XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=seed,
        n_jobs=1,
    )


## 7. Double Machine Learning

DML residualizes both the outcome and treatment:

\[
\tilde{Y}=Y-\widehat{\mathbb{E}}[Y\mid X], \qquad
\tilde{T}=T-\widehat{\mathbb{E}}[T\mid X]
\]

It then estimates how the residualized outcome changes with the residualized treatment. Cross-fitting reduces overfitting bias by generating nuisance predictions out of fold.

Because both `conversion` and `offer` are binary, the nuisance models are classifiers and the corresponding discrete flags are enabled.


In [ ]:
dml = CausalForestDML(
    model_y=make_binary_xgb(),
    model_t=make_binary_xgb(),
    discrete_outcome=True,
    discrete_treatment=True,
    cv=3,
    n_estimators=400,
    min_samples_leaf=20,
    max_samples=0.45,
    random_state=SEED,
    n_jobs=-1,
)

dml_dowhy = dml.dowhy.fit(
    Y_train,
    T_train,
    X=X_train,
    outcome_names=["conversion"],
    treatment_names=["offer"],
    feature_names=list(X_train.columns),
    target_units=X_test,
    inference="auto",
)


### DoWhy identification

DoWhy organizes causal analysis into four stages:

1. **Model** the assumed causal structure.
2. **Identify** the estimand under those assumptions.
3. **Estimate** the causal effect.
4. **Refute or test sensitivity** by perturbing assumptions or data.

Refutation tests do not prove that an estimate is correct. They test whether the result is stable under specific challenges such as placebo treatments, random common causes, or data subsets.


In [ ]:
print(dml_dowhy.identified_estimand_)


In [ ]:
# Optional causal graph generated by the wrapper.
# For observational applications, replace it with a domain-informed DAG.
plt.figure(figsize=(10, 7))
dml_dowhy.view_model()
plt.show()


In [ ]:
cate_dml = np.asarray(dml.effect(X_test)).reshape(-1)
policy_dml_uplift_only = (cate_dml > 0).astype("int8")

pd.Series(
    policy_dml_uplift_only,
    name="DML policy",
).value_counts().sort_index()


## 8. Doubly Robust Learning

A DR learner combines:

- an outcome model, \(\widehat{\mu}_t(X)=\widehat{\mathbb{E}}[Y\mid X,T=t]\)
- a propensity model, \(\widehat{e}(X)=\widehat{\Pr}(T=1\mid X)\)

It constructs a doubly robust pseudo-outcome and learns treatment-effect heterogeneity from it. The estimator is consistent when either the outcome model or the propensity model is correctly specified, subject to the usual identification and regularity assumptions.


In [ ]:
drl = ForestDRLearner(
    model_propensity=make_binary_xgb(),
    model_regression=make_binary_xgb(),
    discrete_outcome=True,
    min_propensity=0.01,
    cv=3,
    n_estimators=400,
    min_samples_leaf=20,
    max_samples=0.45,
    random_state=SEED,
    n_jobs=-1,
)

drl_dowhy = drl.dowhy.fit(
    Y_train,
    T_train,
    X=X_train,
    outcome_names=["conversion"],
    treatment_names=["offer"],
    feature_names=list(X_train.columns),
    target_units=X_test,
    inference="auto",
)


In [ ]:
print(drl_dowhy.identified_estimand_)


In [ ]:
cate_drl = np.asarray(drl.effect(X_test)).reshape(-1)
policy_drl_uplift_only = (cate_drl > 0).astype("int8")

pd.Series(
    policy_drl_uplift_only,
    name="DRL policy",
).value_counts().sort_index()


## 9. Compare the CATE distributions and uplift rankings

The true individual treatment effect is unobserved, so ordinary prediction metrics cannot directly measure CATE accuracy. In a randomized or conditionally ignorable holdout sample, uplift curves, AUUC, and Qini AUC evaluate how well a model ranks customers by incremental response.

Use the same holdout sample for every model.


In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(cate_dml, bins=30, alpha=0.45, label="Causal Forest DML")
plt.hist(cate_drl, bins=30, alpha=0.45, label="Forest DR Learner")
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Distribution of predicted conditional treatment effects")
plt.xlabel("Predicted change in conversion probability")
plt.ylabel("Number of customers")
plt.legend()
plt.show()


In [ ]:
y_test_np = Y_test.to_numpy()
t_test_np = T_test.to_numpy()

uplift_metrics = pd.DataFrame(
    {
        "AUUC": {
            "Causal Forest DML": uplift_auc_score(
                y_true=y_test_np,
                uplift=cate_dml,
                treatment=t_test_np,
            ),
            "Forest DR Learner": uplift_auc_score(
                y_true=y_test_np,
                uplift=cate_drl,
                treatment=t_test_np,
            ),
        },
        "Qini AUC": {
            "Causal Forest DML": qini_auc_score(
                y_true=y_test_np,
                uplift=cate_dml,
                treatment=t_test_np,
            ),
            "Forest DR Learner": qini_auc_score(
                y_true=y_test_np,
                uplift=cate_drl,
                treatment=t_test_np,
            ),
        },
    }
).sort_values("Qini AUC", ascending=False)

uplift_metrics


In [ ]:
plot_uplift_curve(
    y_true=y_test_np,
    uplift=cate_dml,
    treatment=t_test_np,
    perfect=False,
)
plt.title("Uplift curve — Causal Forest DML")
plt.show()


In [ ]:
plot_uplift_curve(
    y_true=y_test_np,
    uplift=cate_drl,
    treatment=t_test_np,
    perfect=False,
)
plt.title("Uplift curve — Forest DR Learner")
plt.show()


### Model-selection guidance

Prefer the model with stronger holdout AUUC/Qini performance **and** stable results across repeated splits or cross-validation. A single uplift curve is not enough to declare one estimator universally more accurate.

For production use, confirm the selected targeting rule in a new randomized holdout experiment.


## 10. Cost-aware targeting and policy value

A positive CATE is not automatically profitable.

If:

- one conversion is worth `price`
- sending an offer costs `offer_cost`
- CATE is the predicted increase in conversion probability

then target a customer only when:

\[
\text{price}\times \widehat{\tau}(X) > \text{offer cost}
\]

or equivalently:

\[
\widehat{\tau}(X) > \frac{\text{offer cost}}{\text{price}}
\]

The policy-value function below uses inverse-propensity weighting on the untouched test sample. With this dataset, the assignment is treated as randomized and a constant propensity is estimated from the training sample. For observational campaigns, replace it with out-of-fold customer-level propensity estimates.


In [ ]:
def cost_aware_policy(
    cate: np.ndarray,
    cost_fraction: float,
) -> np.ndarray:
    # Treat only when expected incremental value exceeds offer cost.
    cate = np.asarray(cate, dtype=float).reshape(-1)
    return (cate > cost_fraction).astype("int8")


def ipw_policy_value(
    policy: np.ndarray,
    outcome: np.ndarray,
    treatment: np.ndarray,
    *,
    price: float,
    offer_cost: float,
    propensity_treated: np.ndarray | float,
) -> float:
    # Estimate mean net value of a deterministic policy using IPW.
    policy = np.asarray(policy, dtype=int).reshape(-1)
    outcome = np.asarray(outcome, dtype=float).reshape(-1)
    treatment = np.asarray(treatment, dtype=int).reshape(-1)

    if not (len(policy) == len(outcome) == len(treatment)):
        raise ValueError(
            "policy, outcome, and treatment must have equal length"
        )

    propensity_treated = np.broadcast_to(
        np.asarray(propensity_treated, dtype=float),
        treatment.shape,
    )
    propensity_treated = np.clip(
        propensity_treated,
        0.01,
        0.99,
    )

    observed_assignment_probability = np.where(
        treatment == 1,
        propensity_treated,
        1.0 - propensity_treated,
    )
    observed_net_value = price * outcome - offer_cost * treatment
    matches_policy = treatment == policy

    weighted_value = (
        matches_policy
        * observed_net_value
        / observed_assignment_probability
    )
    return float(np.mean(weighted_value))


In [ ]:
PRICE = 20.0
COST_FRACTIONS = np.array([0.03, 0.04, 0.05, 0.06])

# Randomized-assignment approximation, estimated from training data only.
test_propensity = np.full(
    len(T_test),
    T_train.mean(),
    dtype=float,
)

policy_values = {}

for cost_fraction in COST_FRACTIONS:
    offer_cost = PRICE * cost_fraction

    policies = {
        "ALL": np.ones(len(Y_test), dtype="int8"),
        "NONE": np.zeros(len(Y_test), dtype="int8"),
        "Causal Forest DML": cost_aware_policy(
            cate_dml,
            cost_fraction,
        ),
        "Forest DR Learner": cost_aware_policy(
            cate_drl,
            cost_fraction,
        ),
    }

    for policy_name, policy in policies.items():
        policy_values.setdefault(policy_name, {})[
            cost_fraction * 100
        ] = ipw_policy_value(
            policy,
            y_test_np,
            t_test_np,
            price=PRICE,
            offer_cost=offer_cost,
            propensity_treated=test_propensity,
        )

    policy_values.setdefault("OBSERVED ASSIGNMENT", {})[
        cost_fraction * 100
    ] = float(
        np.mean(
            PRICE * y_test_np
            - offer_cost * t_test_np
        )
    )

revenue_table = pd.DataFrame(policy_values).T
revenue_table.columns.name = "Offer cost (% of conversion value)"
revenue_table


In [ ]:
ax = revenue_table.T.plot(
    figsize=(12, 7),
    marker="o",
)
ax.set_title("Estimated net value by policy and offer cost")
ax.set_xlabel("Offer cost (% of conversion value)")
ax.set_ylabel("Estimated net value per customer")
ax.legend(
    title="Policy",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.tight_layout()
plt.show()


Interpret the revenue table rather than hard-coding a conclusion. The best policy can change when the promotion cost, conversion value, treatment propensity, or model estimates change.

The IPW evaluation relies on adequate overlap and correct assignment probabilities. Large or unstable weights are a warning sign.


## 11. Interpret treatment-effect heterogeneity

A shallow surrogate tree summarizes broad segments with similar predicted effects. It is an interpretation aid, not the causal estimator itself.

The fitted **EconML estimator** is passed to the interpreter—not the DoWhy wrapper.


In [ ]:
cate_interpreter = SingleTreeCateInterpreter(
    include_model_uncertainty=True,
    max_depth=3,
    min_samples_leaf=100,
    random_state=SEED,
)

cate_interpreter.interpret(drl, X_test)

plt.figure(figsize=(24, 8))
cate_interpreter.plot(
    feature_names=list(X.columns),
    fontsize=10,
)
plt.show()


### Turning the model into a campaign rule

A practical targeting workflow is:

1. Estimate CATE on eligible customers.
2. Convert CATE into expected incremental value.
3. Subtract offer and fulfilment costs.
4. Exclude customers with weak overlap or highly uncertain estimates.
5. Rank the remaining customers by expected net incremental value.
6. Apply budget, capacity, fairness, and contact-frequency constraints.
7. Validate the final rule with a randomized holdout group.

Do not use a single tree split or a positive CATE alone as the final production rule.


## 12. Optional DoWhy refutation tests

These checks challenge a fitted estimate under specific perturbations. They are useful diagnostics, but they do not validate every causal assumption.

```python
placebo_result = drl_dowhy.refute_estimate(
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    random_seed=SEED,
)
print(placebo_result)

subset_result = drl_dowhy.refute_estimate(
    method_name="data_subset_refuter",
    subset_fraction=0.80,
    random_seed=SEED,
)
print(subset_result)
```


## References

- [EconML documentation](https://www.pywhy.org/EconML/)
- [DoWhy documentation](https://www.pywhy.org/dowhy/)
- [EconML causal-forest documentation](https://www.pywhy.org/EconML/_autosummary/econml.dml.CausalForestDML.html)
- [EconML ForestDRLearner documentation](https://www.pywhy.org/EconML/_autosummary/econml.dr.ForestDRLearner.html)
- [Python Causality Handbook — Doubly Robust Estimation](https://matheusfacure.github.io/python-causality-handbook/12-Doubly-Robust-Estimation.html)
